# 5.12 — Particle filters

Particles carry an approximate posterior through nonlinear dynamics by predict, weight, and resample.

Particle filters generalize HMM and Kalman filtering using sampling. They are the sequential Monte Carlo bridge to MCMC.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(512)


def gaussian_pdf(y, mean, std):
    return np.exp(-0.5 * ((y - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def systematic_resample(particles, weights):
    n = len(weights)
    positions = (rng.random() + np.arange(n)) / n
    cumulative = np.cumsum(weights)
    indexes = np.searchsorted(cumulative, positions)
    return particles[indexes]


def effective_sample_size(weights):
    weights = np.asarray(weights, dtype=float)
    return 1.0 / np.sum(weights ** 2)


def particle_filter(particles, observations, transition, likelihood, resample=True):
    particles = np.asarray(particles, dtype=float)
    n = len(particles)
    weights = np.ones(n) / n
    estimates = []
    ess_values = []
    snapshots = []
    for y in observations:
        particles = transition(particles)
        raw_weights = weights * likelihood(y, particles)
        total = raw_weights.sum()
        if total <= 0:
            weights = np.ones(n) / n
        else:
            weights = raw_weights / total
        ess = effective_sample_size(weights)
        estimate = np.average(particles, weights=weights, axis=0)
        estimates.append(estimate)
        ess_values.append(ess)
        snapshots.append((particles.copy(), weights.copy()))
        if resample and ess < n / 2:
            particles = systematic_resample(particles, weights)
            weights = np.ones(n) / n
    return {"estimates": np.asarray(estimates), "ess": np.asarray(ess_values), "snapshots": snapshots}


def grid_reference_1d(observations, grid, transition_std, obs_std):
    belief = np.ones_like(grid) / grid.size
    marginals = []
    dx = grid[1] - grid[0]
    transition = gaussian_pdf(grid[:, None], grid[None, :], transition_std)
    transition = transition / transition.sum(axis=0, keepdims=True)
    for y in observations:
        pred = transition @ belief
        weights = pred * gaussian_pdf(y, grid, obs_std)
        belief = weights / weights.sum()
        marginals.append(belief.copy())
    means = np.array([np.sum(m * grid) for m in marginals])
    return means, marginals


def make_particle_ladder():
    true1 = np.array([0.8, 1.1, 1.0])
    obs1 = true1 + rng.normal(0, 0.25, len(true1))
    true2 = np.array([0.0, 0.3, 0.7, 0.9])
    obs2 = true2 + rng.normal(0, 0.35, len(true2))
    true3 = np.array([-1.0, -0.6, 0.6, 1.0])
    obs3 = true3 ** 2 + rng.normal(0, 0.12, len(true3))
    true4 = np.column_stack([np.linspace(0, 1, 6), np.linspace(1, 0, 6)])
    obs4 = true4 + rng.normal(0, 0.2, true4.shape)
    true5 = np.cumsum(rng.normal(0.05, 0.15, (8, 3)), axis=0)
    obs5 = true5 + rng.normal(0, 0.08, true5.shape)
    return [
        {"name": "D1 1-D exact target", "true": true1, "obs": obs1, "dim": 1, "transition_std": 0.25, "obs_std": 0.3},
        {"name": "D2 4-state particle approximation", "true": true2, "obs": obs2, "dim": 1, "transition_std": 0.30, "obs_std": 0.35},
        {"name": "D3 bimodal nonlinear observation", "true": true3, "obs": obs3, "dim": 1, "transition_std": 0.35, "obs_std": 0.18, "nonlinear": True},
        {"name": "D4 correlated 2-D tracking", "true": true4, "obs": obs4, "dim": 2, "transition_std": 0.22, "obs_std": 0.25},
        {"name": "D5 higher-dim narrow proposal", "true": true5, "obs": obs5, "dim": 3, "transition_std": 0.04, "obs_std": 0.08},
    ]


def run_particle_rung(rung, n_particles=200, proposal_scale=1.0):
    dim = rung["dim"]
    if dim == 1:
        particles = rng.normal(0.0, 1.0, n_particles)
        def transition(x):
            return x + rng.normal(0.0, rung["transition_std"] * proposal_scale, x.shape)
        if rung.get("nonlinear", False):
            def likelihood(y, x):
                return gaussian_pdf(y, x ** 2, rung["obs_std"])
        else:
            def likelihood(y, x):
                return gaussian_pdf(y, x, rung["obs_std"])
    else:
        particles = rng.normal(0.0, 1.0, (n_particles, dim))
        def transition(x):
            return x + rng.normal(0.0, rung["transition_std"] * proposal_scale, x.shape)
        def likelihood(y, x):
            diff = x - y
            return np.exp(-0.5 * np.sum((diff / rung["obs_std"]) ** 2, axis=1))
    return particle_filter(particles, rung["obs"], transition, likelihood, resample=True)


## The concept, built once (D1)

Sequential Monte Carlo updates particle weights by observation likelihood:

$$w_t^{(i)}\propto w_{t-1}^{(i)}p(y_t\mid x_t^{(i)}),\quad \hat p(x_t\mid y_{1:t})\approx\sum_i w_t^{(i)}\delta_{x_t^{(i)}}.$$

The lesson uses $w_i\propto\mathcal N(1.2;x_i,0.3)$ and normalizes weights to sum to 1.

In [ ]:

lesson_particles = np.linspace(0.6, 1.4, 200)
predicted_mean = 0.997563
weights_raw = gaussian_pdf(1.2, lesson_particles, 0.3)
weights = weights_raw / weights_raw.sum()
posterior_mean = float(np.sum(weights * lesson_particles))
ess = effective_sample_size(weights)
print("predicted particle mean", predicted_mean)
print("weight sum", weights.sum())
print("posterior mean", posterior_mean)
print("ESS", ess)
assert np.isclose(predicted_mean, 0.997563)
assert np.isclose(weights.sum(), 1.0)
assert 1.0 < posterior_mean < 1.3
assert ess < 200


The effective sample size $1/\sum_i w_i^2$ reports how many particles are meaningfully contributing before resampling.

In [ ]:

uniform_ess = effective_sample_size(np.ones(200) / 200)
concentrated_ess = ess
print("uniform ESS", uniform_ess)
print("concentrated ESS", concentrated_ess)
assert np.isclose(uniform_ess, 200.0)
assert concentrated_ess < uniform_ess


## The dataset ladder

D1-D5 move from one-dimensional references to bimodal nonlinear observations, correlated 2-D tracking, and a narrow high-dimensional proposal.

In [ ]:

ladder = make_particle_ladder()
for rung in ladder:
    print(rung["name"], "dim", rung["dim"], "steps", len(rung["obs"]))
    print("obs sample", np.round(rung["obs"][:3], 3).tolist())


In [ ]:

rows = []
for rung in ladder:
    result = run_particle_rung(rung, n_particles=200, proposal_scale=1.0)
    estimates = result["estimates"]
    truth = rung["true"]
    if rung["dim"] == 1:
        error = float(np.sqrt(np.mean((estimates - truth) ** 2)))
    else:
        error = float(np.sqrt(np.mean((estimates[:, 0] - truth[:, 0]) ** 2)))
    rows.append({"name": rung["name"], "result": result, "truth": truth, "error": error, "ess": result["ess"]})
for row in rows:
    print(row["name"], "RMSE or marginal error", f"{row['error']:.6f}", "min ESS", f"{row['ess'].min():.1f}")


In [ ]:

fig, axes = plt.subplots(2, len(rows), figsize=(16, 6))
for j, row in enumerate(rows):
    est = row["result"]["estimates"]
    truth = row["truth"]
    if est.ndim == 1:
        axes[0, j].plot(truth, label="true")
        axes[0, j].plot(est, label="particle mean")
        error_curve = np.abs(est - truth)
    else:
        axes[0, j].plot(truth[:, 0], label="true x")
        axes[0, j].plot(est[:, 0], label="particle mean x")
        error_curve = np.abs(est[:, 0] - truth[:, 0])
    axes[0, j].set_title(row["name"].split()[0])
    axes[1, j].plot(row["ess"], label="ESS")
    axes[1, j].plot(error_curve * 200, label="scaled error")
    axes[1, j].set_title("ESS and error")
    axes[1, j].set_xlabel("time")
axes[0, 0].legend()
axes[1, 0].legend()
fig.suptitle("Particle estimates vs truth and ESS/error curve")
plt.tight_layout()


## Pitfall on the hardest rung

When the proposal is too narrow, particles never reach high-likelihood regions. Weights collapse, ESS falls, and resampling duplicates a poor set.

In [ ]:

d5 = ladder[-1]
narrow = run_particle_rung(d5, n_particles=200, proposal_scale=0.15)
adaptive = run_particle_rung(d5, n_particles=200, proposal_scale=2.5)
narrow_min_ess = float(narrow["ess"].min())
adaptive_min_ess = float(adaptive["ess"].min())
narrow_rmse = float(np.sqrt(np.mean((narrow["estimates"][:, 0] - d5["true"][:, 0]) ** 2)))
adaptive_rmse = float(np.sqrt(np.mean((adaptive["estimates"][:, 0] - d5["true"][:, 0]) ** 2)))
print("narrow proposal min ESS", narrow_min_ess)
print("adaptive proposal min ESS", adaptive_min_ess)
print("narrow RMSE", narrow_rmse)
print("adaptive RMSE", adaptive_rmse)
assert narrow_min_ess < 200


## Evaluate it + Practice

- Metric: ESS and marginal or filtering RMSE versus reference truth
- No-skill baseline: unweighted random-walk particles without likelihood weighting
- Cheap sanity check: weights normalize to one and ESS is below N when concentrated
- Ablation: turn off resampling and watch ESS collapse
- Failure signals: min ESS near one, particle impoverishment, or estimates outside likely regions


Practice: Double the number of particles on D3 and compare ESS.

Practice: Make D1 observation noise smaller and inspect weight concentration.

Practice: Change D5 proposal scale and plot RMSE versus ESS.